# LeetCode 666 — Path Sum IV bằng Python

Link đề: https://leetcode.com/problems/path-sum-iv/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Array, Hash Table, Binary Tree

Skeleton LeetCode:

```python
class Solution:
    def pathSum(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đây là bài **Premium** (cần tài khoản trả phí để xem trên LeetCode). Đề trong notebook này được viết theo hiểu biết về bài (có đối chiếu bản mô tả tiếng Anh), nên nếu bạn có tài khoản Premium hãy **đối chiếu lại đề gốc**. Hai ví dụ gốc trên LeetCode có kèm hình vẽ cây; trong notebook ta **vẽ lại bằng chữ**, hãy đối chiếu hình gốc nếu có điều kiện.

Điểm đặc biệt của bài: cây nhị phân **không** cho dưới dạng nút `TreeNode` mà được **mã hóa thành danh sách số nguyên 3 chữ số** (chữ số hàng trăm = độ sâu, hàng chục = vị trí trong tầng, hàng đơn vị = giá trị). Ta sẽ học cách giải mã, rồi đi từ "dựng cây thật rồi DFS" tới "DFS thẳng trên dict", "DFS bằng ngăn xếp", "một lượt quét theo cha", và "đếm đóng góp theo số lá".

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho cây nhị phân độ sâu `< 5` được mã hóa thành mảng các số có 3 chữ số `d p v`; trả về **tổng của tổng mọi đường đi từ gốc tới lá**.
2. Nắm công cụ Python cần dùng: tách chữ số bằng `//` và `%`, `divmod`, dict (bảng băm) với `in`/`get`, biểu thức dict comprehension, đệ quy với `nonlocal`, ngăn xếp bằng list (`append`/`pop`), `reversed`.
3. Nắm cách đánh số cây nhị phân đầy đủ: con của `(d, p)` là `(d+1, 2p-1)` và `(d+1, 2p)`; cha của `(d, p)` là `(d-1, (p+1)//2)`.
4. Đi từ **dựng cây `TreeNode` + DFS** → **DFS trực tiếp trên dict** → **DFS lặp bằng ngăn xếp** → **một lượt theo cha (quét xuôi)** → **đếm đóng góp theo số lá (quét ngược)**.
5. Tránh các lỗi kinh điển: coi nút có **một** con là lá, sai công thức con, cộng tổng giá trị thay vì tổng đường đi, không "quay lui" tổng khi dùng biến toàn cục, và dùng nhầm khóa (có cả chữ số giá trị).

## 2. Đề bài & ví dụ

Nếu độ sâu của một cây nhỏ hơn `5` thì cây đó có thể biểu diễn bằng một mảng các số nguyên **3 chữ số**. Cho mảng `nums` **tăng dần**, mỗi số mô tả một nút:

- **chữ số hàng trăm** là độ sâu `d` của nút, `1 <= d <= 4` (gốc có `d = 1`);
- **chữ số hàng chục** là vị trí `p` của nút trong tầng của nó, `1 <= p <= 8`, tính theo **cây nhị phân đầy đủ** (vị trí đánh số từ trái sang phải);
- **chữ số hàng đơn vị** là giá trị `v` của nút, `0 <= v <= 9`.

Hãy trả về **tổng của tất cả các đường đi từ gốc tới lá** (tổng của mỗi đường là tổng giá trị các nút trên đường đó). Đề đảm bảo mảng biểu diễn một cây nhị phân hợp lệ, liên thông.

**Ví dụ 1**

- Input: `nums = [113, 215, 221]`
- Output: `12`
- Giải thích: gốc `3` (độ sâu 1, vị trí 1); con trái `5` (độ sâu 2, vị trí 1); con phải `1` (độ sâu 2, vị trí 2). Hai đường: `3 + 5 = 8` và `3 + 1 = 4`; tổng `8 + 4 = 12`.

**Ví dụ 2**

- Input: `nums = [113, 221]`
- Output: `4`
- Giải thích: gốc `3` chỉ có con **phải** `1`. Chỉ có một đường `3 + 1 = 4`.

**Ràng buộc:**

- `1 <= nums.length <= 15`
- `110 <= nums[i] <= 489`
- `nums` biểu diễn một cây nhị phân hợp lệ có độ sâu nhỏ hơn 5.
- `nums` sắp xếp tăng dần.

Vì tối đa 4 tầng nên cây có tối đa `1 + 2 + 4 + 8 = 15` nút — rất nhỏ. Bài này không khó về hiệu năng; nó khó ở **cách đọc mã hóa** và **định nghĩa "lá"**.

In [1]:
from typing import List
import random

examples = [
    ([113, 215, 221], 12),
    ([113, 221], 4),
]
for nums, expected in examples:
    print("nums =", nums, "| kỳ vọng:", expected)

nums = [113, 215, 221] | kỳ vọng: 12
nums = [113, 221] | kỳ vọng: 4


## 3. Kiến thức nền

### 3.1. Giải mã số 3 chữ số bằng `//` và `%`

Với `num = 215`:

- `num // 100 = 2` (chia nguyên, bỏ phần dư) → chữ số **hàng trăm**;
- `num // 10 % 10 = 1` → chữ số **hàng chục** (bỏ chữ số cuối rồi lấy phần dư khi chia 10);
- `num % 10 = 5` → chữ số **hàng đơn vị**.

Còn `num // 10 = 21` là **mã nút** ghép `(độ sâu, vị trí)` thành số có hai chữ số; ta sẽ dùng nó làm **khóa**. `divmod(21, 10)` trả cặp `(2, 1)` — thương và dư — tức tách khóa thành `(d, p)` trong một lần.

In [2]:
num = 215
print("hàng trăm :", num // 100)
print("hàng chục :", num // 10 % 10)
print("hàng đơn vị:", num % 10)
print("mã nút num // 10 =", num // 10, "| divmod(21, 10) =", divmod(21, 10))
print("kiểm tra: giá trị phép chia", 7 // 2, 7 % 2, divmod(7, 2))

hàng trăm : 2
hàng chục : 1
hàng đơn vị: 5
mã nút num // 10 = 21 | divmod(21, 10) = (2, 1)
kiểm tra: giá trị phép chia 3 1 (3, 1)


### 3.2. Dict (bảng băm): khóa → giá trị

Ta lưu **mã nút → giá trị** vào một `dict` để tra cứu nhanh bất kỳ nút nào. Toán tử `key in d` kiểm tra có khóa hay không trong `O(1)` trung bình. Dict comprehension `{k: v for ...}` dựng dict trong một dòng. Tra khóa **không có** bằng `d[key]` sẽ báo `KeyError`, nên ta dùng `in` hoặc `d.get(key)` (trả `None` nếu không có).

In [3]:
nums = [113, 215, 221]
mp = {num // 10: num % 10 for num in nums}
print(mp)
print("11 in mp:", 11 in mp, "| 31 in mp:", 31 in mp, "| mp.get(31):", mp.get(31), "| mp.get(21):", mp.get(21))
try:
    mp[31]
except KeyError as e:
    print("mp[31] -> KeyError", e)

{11: 3, 21: 5, 22: 1}
11 in mp: True | 31 in mp: False | mp.get(31): None | mp.get(21): 5
mp[31] -> KeyError 31


### 3.3. Cây nhị phân **đầy đủ** và công thức con/cha

Trong cây nhị phân đầy đủ, tầng `d` có `2^(d-1)` vị trí, đánh số `1 .. 2^(d-1)` từ trái sang phải. Vị trí `p` ở tầng `d` có hai con ở tầng `d + 1`:

- con trái: vị trí `2p - 1`;
- con phải: vị trí `2p`.

Và cha của `(d, p)` ở tầng `d - 1` có vị trí `(p + 1) // 2`. Nút có thể "vắng mặt" trong `nums` — khi đó trong cây thật nó không tồn tại, nhưng **vị trí vẫn được giữ** theo cây đầy đủ.

Với mã nút `key = d * 10 + p`, con trái là `(d + 1) * 10 + 2p - 1` và con phải bằng con trái cộng 1.

In [4]:
def child_ids(key):
    d, p = divmod(key, 10)
    left = (d + 1) * 10 + 2 * p - 1
    return left, left + 1


def parent_id(key):
    d, p = divmod(key, 10)
    return (d - 1) * 10 + (p + 1) // 2


for d in range(1, 4):
    for p in range(1, 2 ** (d - 1) + 1):
        key = d * 10 + p
        print(f"nút {key} (tầng {d}, vị trí {p}): con = {child_ids(key)}", end="")
        print(f" | cha = {parent_id(key) if d > 1 else '-'}")

nút 11 (tầng 1, vị trí 1): con = (21, 22) | cha = -
nút 21 (tầng 2, vị trí 1): con = (31, 32) | cha = 11
nút 22 (tầng 2, vị trí 2): con = (33, 34) | cha = 11
nút 31 (tầng 3, vị trí 1): con = (41, 42) | cha = 21
nút 32 (tầng 3, vị trí 2): con = (43, 44) | cha = 21
nút 33 (tầng 3, vị trí 3): con = (45, 46) | cha = 22
nút 34 (tầng 3, vị trí 4): con = (47, 48) | cha = 22


Kiểm tra tính nhất quán: cha của con trái và con phải của một nút phải chính là nút đó.

In [5]:
for d in range(1, 4):
    for p in range(1, 2 ** (d - 1) + 1):
        key = d * 10 + p
        for c in child_ids(key):
            assert parent_id(c) == key
print("cha(con(x)) == x với mọi nút tầng 1..3")

cha(con(x)) == x với mọi nút tầng 1..3


### 3.4. Lá, đường đi gốc → lá, và DFS có truyền tổng

- **Lá** (leaf) là nút **không có con trái và không có con phải** — cả hai đều vắng. Nút chỉ có **một** con **không phải lá**.
- **DFS** (duyệt theo chiều sâu): từ một nút, đi xuống từng con. Ta truyền theo **tổng đường đi từ gốc tới nút hiện tại** `t` qua tham số — mỗi lời gọi đệ quy có bản sao `t` riêng, nên khi quay về không cần "trừ lại".
- Khi tới lá, cộng `t` vào đáp án.

Ta dùng `nonlocal` để hàm đệ quy lồng bên trong có thể **gán lại** biến `ans` của hàm bao ngoài.

In [6]:
def demo_nonlocal():
    ans = 0
    def add(x):
        nonlocal ans          # nếu thiếu dòng này, ans += x sẽ báo UnboundLocalError
        ans += x
    add(3)
    add(4)
    return ans


print(demo_nonlocal())

7


### 3.5. Ngăn xếp bằng list và `reversed`

Ngăn xếp (stack) vào-sau-ra-trước: `stack.append(x)` đẩy vào cuối, `stack.pop()` lấy ra phần tử cuối. Ta dùng nó để thay đệ quy bằng vòng lặp (cách 3). Hàm `reversed(nums)` duyệt mảng từ **cuối về đầu** (cách 5).

In [7]:
stack = []
stack.append("a")
stack.append("b")
stack.append("c")
print("pop:", stack.pop(), stack.pop(), "| còn lại:", stack)
print(list(reversed([1, 2, 3])))

pop: c b | còn lại: ['a']
[3, 2, 1]


## 4. Ý tưởng nền tảng / trực giác

Vẽ cây của ví dụ 1 `nums = [113, 215, 221]`:

```
          3          (tầng 1, vị trí 1)  = mã 11
        /   \
       5     1       (tầng 2, vị trí 1 và 2) = mã 21, 22
```

Hai nút ở tầng 2 đều là lá (không có nút tầng 3 nào). Hai đường: `3 → 5` (tổng 8) và `3 → 1` (tổng 4). Đáp án `8 + 4 = 12`.

Ví dụ 2 `[113, 221]`: tầng 1 có `3`; tầng 2 chỉ có nút vị trí **2** (`221`): con **phải**.

```
          3          (mã 11)
            \
             1       (mã 22)
```

Chỉ một lá (`1`), một đường `3 + 1 = 4`. Chú ý: gốc `3` **không phải lá** dù thiếu con trái, vì nó vẫn có con phải.

Trực giác cho thuật toán: đi từ gốc xuống, **mang theo tổng đường đi** đã tích lũy; khi chạm lá thì "chốt" tổng đó. Muốn biết nút `(d, p)` có con hay không, ta tra mã `(d+1, 2p-1)` và `(d+1, 2p)` trong dict.

In [8]:
def draw(nums):
    """In cây theo từng tầng (chỉ nút có mặt) để quan sát."""
    mp = {num // 10: num % 10 for num in nums}
    for d in range(1, 5):
        row = []
        for p in range(1, 2 ** (d - 1) + 1):
            row.append(str(mp[d * 10 + p]) if d * 10 + p in mp else ".")
        if any(c != "." for c in row):
            print(f"tầng {d}: {' '.join(row)}")


draw([113, 215, 221])
print()
draw([113, 221])
print()
draw([113, 215, 221, 314, 326])

tầng 1: 3
tầng 2: 5 1

tầng 1: 3
tầng 2: . 1

tầng 1: 3
tầng 2: 5 1
tầng 3: 4 6 . .


## 5. Cách 1 — Dựng cây `TreeNode` thật rồi DFS

### Ý tưởng

Cách "quen thuộc nhất": biến mã hóa thành cây nút `TreeNode` (với `val`, `left`, `right`) rồi giải bài **tổng đường đi gốc → lá** quen thuộc. Các bước:

1. Tạo một nút `TreeNode` cho mỗi số, lưu vào dict theo mã nút `num // 10`.
2. Với mỗi nút, tra con trái/phải trong dict theo công thức mục 3.3 và nối con trỏ.
3. DFS từ gốc (mã `11`), truyền tổng đường đi; khi gặp lá trả về tổng đường đó, ngược lại trả về **tổng** các kết quả của con.

Notebook không cần `TreeNode` có sẵn, nên ta tự định nghĩa.

In [9]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(nums):
    nodes = {num // 10: TreeNode(num % 10) for num in nums}
    for key, node in nodes.items():
        l, r = child_ids(key)
        node.left = nodes.get(l)
        node.right = nodes.get(r)
    return nodes.get(11)


def path_sum_tree(nums):
    root = build_tree(nums)

    def dfs(node, t):
        t += node.val
        if node.left is None and node.right is None:
            return t                         # lá: chốt tổng đường đi
        total = 0
        if node.left is not None:
            total += dfs(node.left, t)
        if node.right is not None:
            total += dfs(node.right, t)
        return total

    return dfs(root, 0)


for nums, expected in examples:
    print(nums, "->", path_sum_tree(nums), "| kỳ vọng", expected)
    assert path_sum_tree(nums) == expected

[113, 215, 221] -> 12 | kỳ vọng 12
[113, 221] -> 4 | kỳ vọng 4


### Độ phức tạp và nút thắt

- Dựng cây `O(n)`, DFS `O(n)` (mỗi nút thăm một lần): tổng `O(n)` thời gian, `O(n)` bộ nhớ cho các nút (và `O(h)` ngăn xếp đệ quy với `h <= 4`).
- Với `n <= 15` hiệu năng là chuyện nhỏ. **Nút thắt thật** là sự rườm rà: ta dựng cả một cấu trúc cây chỉ để đọc lại nó. Dict `mã → giá trị` đã chứa đủ thông tin; ta có thể DFS **thẳng trên dict** (cách 2).

Ta cũng cần một **oracle độc lập** để kiểm thử. Ta sẽ dùng cách đánh số kiểu **heap**: nút `(d, p)` có chỉ số `h = 2^(d-1) + p - 1`, con là `2h` và `2h + 1`, cha là `h // 2`. Với mỗi lá, **leo ngược** lên gốc cộng giá trị. Cách này không dùng công thức `(d+1, 2p-1)` nên đủ độc lập với các cách giải.

In [10]:
def heap_index(num):
    d, p = num // 100, num // 10 % 10
    return 2 ** (d - 1) + p - 1


def oracle_paths(nums):
    """Liệt kê mọi đường gốc -> lá dưới dạng list giá trị (từ gốc tới lá), dùng chỉ số heap."""
    val = {heap_index(num): num % 10 for num in nums}
    paths = []
    for h in sorted(val):
        if 2 * h not in val and 2 * h + 1 not in val:      # lá
            path = []
            x = h
            while x >= 1:
                path.append(val[x])
                x //= 2
            paths.append(path[::-1])
    return paths


def oracle_path_sum(nums):
    return sum(sum(path) for path in oracle_paths(nums))


for nums, expected in examples:
    print(nums, "| các đường:", oracle_paths(nums), "| tổng:", oracle_path_sum(nums))
    assert oracle_path_sum(nums) == expected

[113, 215, 221] | các đường: [[3, 5], [3, 1]] | tổng: 12
[113, 221] | các đường: [[3, 1]] | tổng: 4


## 6. Ý tưởng thuật toán / bước đột phá

Ta **không cần dựng cây**. Dict `mp = {num // 10: num % 10}` đã cho biết:

1. Nút nào tồn tại (`key in mp`) và giá trị của nó (`mp[key]`).
2. Con của nút `key` là `(d+1, 2p-1)` và `(d+1, 2p)`; nó có con nếu ít nhất một trong hai mã nằm trong `mp`.

Vậy: **DFS từ mã `11`**, mang theo tổng `t`. Tại nút `key`: `t += mp[key]`; nếu **cả hai con vắng** thì đây là lá → `ans += t`; ngược lại đệ quy vào các con (hàm tự ngắt nếu con không có trong `mp`).

Hai biến thể cùng ý tưởng: thay đệ quy bằng ngăn xếp (cách 3); hoặc tận dụng việc `nums` **tăng dần** (cha luôn đứng trước con) để tính tổng đường đi **xuôi một lượt** (cách 4) hay đếm đóng góp **ngược một lượt** (cách 5).

## 7. Các cách tiếp theo

Tổng cộng 5 cách: (1) dựng cây `TreeNode` + DFS (mục 5), (2) DFS thẳng trên dict, (3) DFS lặp bằng ngăn xếp, (4) quét xuôi theo cha, (5) quét ngược đếm đóng góp theo số lá.

### 7.1. Cách 2 — DFS đệ quy trực tiếp trên dict

**Giải thích từng đoạn.**

1. `mp = {num // 10: num % 10 for num in nums}`.
2. `dfs(node, t)`: nếu `node not in mp` thì thoát (nút vắng).
3. `t += mp[node]`; tính `l`, `r` theo công thức; nếu `l not in mp and r not in mp` thì `ans += t` và thoát.
4. Ngược lại gọi `dfs(l, t)` và `dfs(r, t)`. Một con vắng sẽ tự thoát ở bước 2.

Chú ý điều kiện lá dùng **`and`**: lá phải thiếu **cả hai** con.

In [11]:
def path_sum_dfs(nums):
    mp = {num // 10: num % 10 for num in nums}
    ans = 0

    def dfs(node, t):
        nonlocal ans
        if node not in mp:
            return
        t += mp[node]
        l, r = child_ids(node)
        if l not in mp and r not in mp:      # lá: THIẾU CẢ HAI con
            ans += t
            return
        dfs(l, t)
        dfs(r, t)

    dfs(11, 0)
    return ans


for nums, expected in examples:
    print(nums, "->", path_sum_dfs(nums))
    assert path_sum_dfs(nums) == expected

[113, 215, 221] -> 12
[113, 221] -> 4


**Code nộp LeetCode (cách 2):**

```python
class Solution:
    def pathSum(self, nums: List[int]) -> int:
        def dfs(node, t):
            nonlocal ans
            if node not in mp:
                return
            t += mp[node]
            d, p = divmod(node, 10)
            l = (d + 1) * 10 + (p * 2) - 1
            r = l + 1
            if l not in mp and r not in mp:
                ans += t
                return
            dfs(l, t)
            dfs(r, t)

        ans = 0
        mp = {num // 10: num % 10 for num in nums}
        dfs(11, 0)
        return ans
```

**Độ phức tạp:** Time `O(n)`, Space `O(n)` (dict) cộng `O(h)` cho đệ quy với `h <= 4`.

### 7.2. Cách 3 — DFS lặp bằng ngăn xếp

**Giải thích từng đoạn.**

1. Ngăn xếp chứa các cặp `(mã nút, tổng đường đi tới cha)`; khởi đầu `[(11, 0)]`.
2. Lấy ra một cặp, cộng giá trị nút vào `t`.
3. Nếu nút là lá (cả hai con vắng), `ans += t`.
4. Ngược lại đẩy các con **có mặt** vào ngăn xếp cùng `t` mới.

Không dùng đệ quy nên không lo giới hạn đệ quy (ở đây vô hại vì `h <= 4`, nhưng là kỹ năng cần có cho cây sâu).

In [12]:
def path_sum_stack(nums):
    mp = {num // 10: num % 10 for num in nums}
    ans = 0
    stack = [(11, 0)]
    while stack:
        node, t = stack.pop()
        t += mp[node]
        l, r = child_ids(node)
        if l not in mp and r not in mp:
            ans += t
        else:
            for c in (l, r):
                if c in mp:
                    stack.append((c, t))
    return ans


for nums, expected in examples:
    print(nums, "->", path_sum_stack(nums))
    assert path_sum_stack(nums) == expected

[113, 215, 221] -> 12
[113, 221] -> 4


**Code nộp LeetCode (cách 3):**

```python
class Solution:
    def pathSum(self, nums: List[int]) -> int:
        mp = {num // 10: num % 10 for num in nums}
        ans = 0
        stack = [(11, 0)]
        while stack:
            node, t = stack.pop()
            t += mp[node]
            d, p = divmod(node, 10)
            l = (d + 1) * 10 + 2 * p - 1
            r = l + 1
            if l not in mp and r not in mp:
                ans += t
            else:
                for c in (l, r):
                    if c in mp:
                        stack.append((c, t))
        return ans
```

**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### 7.3. Cách 4 — Quét xuôi theo cha (tận dụng `nums` tăng dần)

**Ý tưởng.** Vì `nums` sắp xếp tăng dần, các nút xuất hiện theo **thứ tự tầng rồi vị trí**, nên **cha luôn đứng trước con**. Do đó một lượt quét đủ:

1. Với nút `(d, p)`: nếu `d = 1` thì `acc[key] = v`; ngược lại tìm cha `(d - 1, (p + 1) // 2)`, đặt `acc[key] = acc[cha] + v` và **đánh dấu cha có con**.
2. `acc[key]` là tổng đường đi từ gốc tới nút `key`.
3. Đáp án là tổng `acc[key]` của mọi nút **chưa từng bị đánh dấu là cha** (tức là lá).

Cách này **không dùng công thức con** mà dùng công thức **cha**; không đệ quy, không ngăn xếp.

In [13]:
def path_sum_parent(nums):
    acc = {}                  # acc[key] = tổng đường đi từ gốc tới nút key
    has_child = set()
    for num in nums:          # tăng dần: cha luôn được xử lý trước con
        key, v = num // 10, num % 10
        d, p = divmod(key, 10)
        if d == 1:
            acc[key] = v
        else:
            parent = (d - 1) * 10 + (p + 1) // 2
            acc[key] = acc[parent] + v
            has_child.add(parent)
    return sum(a for key, a in acc.items() if key not in has_child)


for nums, expected in examples:
    print(nums, "->", path_sum_parent(nums))
    assert path_sum_parent(nums) == expected

[113, 215, 221] -> 12
[113, 221] -> 4


**Code nộp LeetCode (cách 4):**

```python
class Solution:
    def pathSum(self, nums: List[int]) -> int:
        acc = {}
        has_child = set()
        for num in nums:
            key, v = num // 10, num % 10
            d, p = divmod(key, 10)
            if d == 1:
                acc[key] = v
            else:
                parent = (d - 1) * 10 + (p + 1) // 2
                acc[key] = acc[parent] + v
                has_child.add(parent)
        return sum(a for key, a in acc.items() if key not in has_child)
```

**Độ phức tạp:** Time `O(n)`, Space `O(n)`. Cách này dựa vào việc `nums` **tăng dần** (đề bảo đảm); nếu mảng không được sắp xếp thì phải sắp xếp trước.

### 7.4. Cách 5 — Quét ngược: mỗi nút đóng góp `giá trị × số lá trong cây con`

**Ý tưởng khác hẳn.** Tổng của mọi tổng đường đi = **tổng theo nút** của (giá trị nút) × (số đường đi qua nút đó). Số đường gốc→lá **đi qua** nút `x` chính bằng **số lá trong cây con của `x`**. Vậy:

> đáp án = Σ `v(x) × leaves(x)`.

`leaves(x)` tính từ dưới lên: nếu `x` không có con thì `leaves(x) = 1`; ngược lại `leaves(x) = leaves(con trái) + leaves(con phải)` (con vắng tính 0). Do `nums` tăng dần, **duyệt ngược** (từ tầng sâu về gốc) bảo đảm con được xử lý trước cha.

Ví dụ `[113, 215, 221]`: `leaves(21) = 1`, `leaves(22) = 1`, `leaves(11) = 2`. Đáp án `3·2 + 5·1 + 1·1 = 12`.

In [14]:
def path_sum_leaves(nums):
    leaves = {}                         # leaves[key] = số lá trong cây con gốc key
    for num in reversed(nums):          # từ tầng sâu về gốc
        key = num // 10
        l, r = child_ids(key)
        c = leaves.get(l, 0) + leaves.get(r, 0)
        leaves[key] = c if c > 0 else 1
    return sum((num % 10) * leaves[num // 10] for num in nums)


for nums, expected in examples:
    print(nums, "->", path_sum_leaves(nums))
    assert path_sum_leaves(nums) == expected

[113, 215, 221] -> 12
[113, 221] -> 4


**Code nộp LeetCode (cách 5):**

```python
class Solution:
    def pathSum(self, nums: List[int]) -> int:
        leaves = {}
        for num in reversed(nums):
            key = num // 10
            d, p = divmod(key, 10)
            l = (d + 1) * 10 + 2 * p - 1
            c = leaves.get(l, 0) + leaves.get(l + 1, 0)
            leaves[key] = c if c > 0 else 1
        return sum((num % 10) * leaves[num // 10] for num in nums)
```

**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

Năm cách cho cùng đáp án; mục 12 sẽ đối chiếu chúng với `oracle_path_sum` trên **mọi hình dạng cây hợp lệ**.

## 8. TRACE từng bước

Trace cách 2 (DFS trên dict) trên ví dụ 1. Bảng tay:

| bước | nút (mã) | `t` sau khi cộng | con trái / phải có? | hành động | `ans` |
|---|---|---|---|---|---|
| 1 | `11` (giá trị 3) | 3 | `21` có, `22` có | đi xuống con | 0 |
| 2 | `21` (giá trị 5) | 8 | `31` không, `32` không | **lá**, cộng 8 | 8 |
| 3 | `22` (giá trị 1) | 4 | `33` không, `34` không | **lá**, cộng 4 | 12 |

Hàm `trace_dfs` in cây gọi đệ quy thụt lề để thấy rõ "đi xuống" và "quay lại".

In [15]:
def trace_dfs(nums):
    mp = {num // 10: num % 10 for num in nums}
    ans = 0

    def dfs(node, t, indent):
        nonlocal ans
        if node not in mp:
            print(f"{indent}nút {node}: vắng -> quay lại")
            return
        t += mp[node]
        l, r = child_ids(node)
        d, p = divmod(node, 10)
        mark_l = "có" if l in mp else "vắng"
        mark_r = "có" if r in mp else "vắng"
        print(f"{indent}vào nút {node} (tầng {d}, vị trí {p}) giá trị {mp[node]} -> t = {t}; "
              f"con trái {l}: {mark_l}, con phải {r}: {mark_r}")
        if l not in mp and r not in mp:
            ans += t
            print(f"{indent}  LÁ: cộng {t} -> ans = {ans}")
            return
        dfs(l, t, indent + "    ")
        dfs(r, t, indent + "    ")

    dfs(11, 0, "")
    return ans


print("kết quả:", trace_dfs([113, 215, 221]))

vào nút 11 (tầng 1, vị trí 1) giá trị 3 -> t = 3; con trái 21: có, con phải 22: có
    vào nút 21 (tầng 2, vị trí 1) giá trị 5 -> t = 8; con trái 31: vắng, con phải 32: vắng
      LÁ: cộng 8 -> ans = 8
    vào nút 22 (tầng 2, vị trí 2) giá trị 1 -> t = 4; con trái 33: vắng, con phải 34: vắng
      LÁ: cộng 4 -> ans = 12
kết quả: 12


Trace ca có **nút chỉ có một con** (ví dụ 2): gốc có con phải mà vắng con trái — ta thấy ngay gốc **không** bị coi là lá, và lời gọi vào con trái vắng thì quay lại không làm gì.

In [16]:
print("kết quả:", trace_dfs([113, 221]))

vào nút 11 (tầng 1, vị trí 1) giá trị 3 -> t = 3; con trái 21: vắng, con phải 22: có
    nút 21: vắng -> quay lại
    vào nút 22 (tầng 2, vị trí 2) giá trị 1 -> t = 4; con trái 33: vắng, con phải 34: vắng
      LÁ: cộng 4 -> ans = 4
kết quả: 4


Và ca 5 nút `[113, 215, 221, 314, 326]`: gốc `3`; con trái `5` có hai con lá `4` và `6`; con phải `1` là lá. Ba đường: `3+5+4 = 12`, `3+5+6 = 14`, `3+1 = 4`; tổng `30`.

In [17]:
res = trace_dfs([113, 215, 221, 314, 326])
print("kết quả:", res)
assert res == 30 == oracle_path_sum([113, 215, 221, 314, 326])

vào nút 11 (tầng 1, vị trí 1) giá trị 3 -> t = 3; con trái 21: có, con phải 22: có
    vào nút 21 (tầng 2, vị trí 1) giá trị 5 -> t = 8; con trái 31: có, con phải 32: có
        vào nút 31 (tầng 3, vị trí 1) giá trị 4 -> t = 12; con trái 41: vắng, con phải 42: vắng
          LÁ: cộng 12 -> ans = 12
        vào nút 32 (tầng 3, vị trí 2) giá trị 6 -> t = 14; con trái 43: vắng, con phải 44: vắng
          LÁ: cộng 14 -> ans = 26
    vào nút 22 (tầng 2, vị trí 2) giá trị 1 -> t = 4; con trái 33: vắng, con phải 34: vắng
      LÁ: cộng 4 -> ans = 30
kết quả: 30


## 9. Code nộp LeetCode

Đây là cách 2 (DFS trực tiếp trên dict) — ngắn nhất và là cách phổ biến.

```python
from typing import List

class Solution:
    def pathSum(self, nums: List[int]) -> int:
        def dfs(node, t):
            nonlocal ans
            if node not in mp:
                return
            t += mp[node]
            d, p = divmod(node, 10)
            l = (d + 1) * 10 + (p * 2) - 1
            r = l + 1
            if l not in mp and r not in mp:
                ans += t
                return
            dfs(l, t)
            dfs(r, t)

        ans = 0
        mp = {num // 10: num % 10 for num in nums}
        dfs(11, 0)
        return ans
```

In [18]:
class Solution:
    def pathSum(self, nums: List[int]) -> int:
        def dfs(node, t):
            nonlocal ans
            if node not in mp:
                return
            t += mp[node]
            d, p = divmod(node, 10)
            l = (d + 1) * 10 + (p * 2) - 1
            r = l + 1
            if l not in mp and r not in mp:
                ans += t
                return
            dfs(l, t)
            dfs(r, t)

        ans = 0
        mp = {num // 10: num % 10 for num in nums}
        dfs(11, 0)
        return ans


sol = Solution()
for nums, expected in examples:
    got = sol.pathSum(nums)
    print(nums, "->", got, "| kỳ vọng", expected)
    assert got == expected

[113, 215, 221] -> 12 | kỳ vọng 12
[113, 221] -> 4 | kỳ vọng 4


## 10. Vì sao đúng

**Phát biểu.** DFS trả về đúng tổng của tổng mọi đường gốc → lá.

**Chứng minh ngắn.**

1. `mp` chứa đúng các nút của cây (khóa = `(độ sâu, vị trí)`), và theo cây nhị phân đầy đủ, nút `(d, p)` có con tiềm năng là `(d+1, 2p-1)` và `(d+1, 2p)`. Một nút có mặt trong `mp` là một nút của cây; vắng mặt nghĩa là không tồn tại.
2. Với nút `x` được thăm, tham số `t` bằng **tổng giá trị trên đường duy nhất từ gốc tới `x`** (quy nạp: ở gốc `t = 0 + mp[11]`; khi đi từ `x` sang con `c`, `t_c = t_x + mp[c]`).
3. `x` là lá ⇔ **cả hai** con vắng khỏi `mp`. Khi đó `t` là tổng của đúng một đường gốc → lá, và ta cộng nó vào `ans` đúng một lần (mỗi lá thăm đúng một lần, vì cây có đường đi duy nhất từ gốc tới mỗi nút).
4. Nếu `x` không phải lá, ta đệ quy vào các con có mặt; con vắng thoát ngay (`node not in mp`) nên không cộng gì.
5. Mọi lá đều được thăm (đề bảo đảm cây liên thông từ gốc `11`), nên `ans` = Σ (tổng đường gốc → lá).

**Với cách 4:** `acc[key]` là tổng đường tới `key`, đúng theo quy nạp trên thứ tự tăng dần; lá là nút không phải cha của nút nào → tổng `acc` trên lá chính là đáp án.

**Với cách 5:** mỗi đường gốc → lá đi qua nút `x` đúng khi lá của nó nằm trong cây con của `x`; nên giá trị `v(x)` xuất hiện trong đúng `leaves(x)` tổng đường đi.

## 11. Time/Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| oracle (đánh số heap, leo ngược từ mỗi lá) | `O(n * h)` | `O(n)` | độc lập với các cách khác |
| 1. Dựng `TreeNode` + DFS | `O(n)` | `O(n)` | quen thuộc, hơi rườm rà |
| 2. DFS trên dict (`Solution`) | `O(n)` | `O(n)` | cách nộp |
| 3. DFS lặp bằng ngăn xếp | `O(n)` | `O(n)` | không đệ quy |
| 4. Quét xuôi theo cha | `O(n)` | `O(n)` | dựa vào `nums` tăng dần |
| 5. Quét ngược đếm lá | `O(n)` | `O(n)` | góc nhìn "đóng góp" |

Vì `n <= 15` và `h <= 4`, mọi cách chạy tức thì; sự khác biệt giữa các cách là về **độ rõ ràng** và **độ tổng quát** chứ không phải tốc độ. (Với cây sâu hơn, DFS đệ quy có thể gặp giới hạn đệ quy của Python — khi đó ngăn xếp/quét xuôi là lựa chọn tốt hơn.)

## 12. Bộ test

Chiến lược: (a) ví dụ của đề, (b) edge case (chỉ có gốc, chuỗi trái, chuỗi phải, cây đầy đủ), (c) phản ví dụ cho các lời giải sai, (d) **kiểm tra vét cạn mọi hình dạng cây hợp lệ** (có đúng 676 hình dạng với độ sâu `<= 4`) và (e) ngẫu nhiên có seed.

### 12.1. Ví dụ của đề và edge case

In [19]:
ALL = {
    "tree": path_sum_tree,
    "dfs": path_sum_dfs,
    "stack": path_sum_stack,
    "parent": path_sum_parent,
    "leaves": path_sum_leaves,
    "solution": lambda nums: Solution().pathSum(nums),
}

cases = [
    ([113, 215, 221], 12),                          # ví dụ 1
    ([113, 221], 4),                                # ví dụ 2 (gốc chỉ có con phải)
    ([113], 3),                                     # chỉ có gốc: gốc là lá
    ([110], 0),                                     # gốc giá trị 0
    ([111, 211, 311, 411], 4),                      # chuỗi trái dài 4: một đường 1+1+1+1
    ([111, 222, 344, 488], 1 + 2 + 4 + 8),          # chuỗi phải dài 4 (vị trí 1, 2, 4, 8)
    ([113, 215, 221, 314, 326], 30),                # cây lệch 5 nút
    ([113, 215, 221, 314], 12 + 4),                 # con trái 5 chỉ có một con (31): lá là 4 và 1
]
# cây đầy đủ 15 nút, mọi giá trị 1: 8 lá, mỗi đường 4 nút -> 8 * 4 = 32
full_ones = [d * 100 + p * 10 + 1 for d in range(1, 5) for p in range(1, 2 ** (d - 1) + 1)]
assert len(full_ones) == 15
cases.append((full_ones, 32))

for nums, expected in cases:
    assert oracle_path_sum(nums) == expected, (nums, oracle_path_sum(nums), expected)
    for name, fn in ALL.items():
        got = fn(nums)
        assert got == expected, (name, nums, got, expected)
print("tất cả", len(cases), "ca cố định đều đúng với", len(ALL), "cách")

tất cả 9 ca cố định đều đúng với 6 cách


### 12.2. Phản ví dụ cho các lời giải sai

Chi tiết ở mục 13; ở đây ghi lại những ca làm lộ lỗi để dùng làm test hồi quy:

- `[113, 221]` → đúng `4` (gốc chỉ có **một** con thì không phải lá).
- `[113, 215, 221]` → đúng `12` (không phải `9` là tổng mọi giá trị, cũng không phải `6` là tổng giá trị các lá).
- `[113, 215, 221, 314, 326]` → đúng `30` (cho thấy phải "quay lui" tổng khi đổi nhánh).

In [20]:
for nums, expected in [([113, 221], 4), ([113, 215, 221], 12), ([113, 215, 221, 314, 326], 30)]:
    assert oracle_path_sum(nums) == expected
    for name, fn in ALL.items():
        assert fn(nums) == expected, (name, nums)
print("ca hồi quy đều đúng")

ca hồi quy đều đúng


### 12.3. Vét cạn: mọi hình dạng cây hợp lệ (676 hình dạng)

Hình dạng cây hợp lệ với độ sâu `<= 4` được dựng đệ quy: tại nút `(d, p)`, mỗi con có thể **vắng** hoặc **có mặt với một cây con tùy ý**. Số hình dạng khác nhau với gốc có mặt: `f(4) = 1`, `f(3) = (1 + f(4))^2 = 4`, `f(2) = (1 + 4)^2 = 25`, `f(1) = (1 + 25)^2 = 676`. Với mỗi hình dạng ta gán giá trị ngẫu nhiên 3 lần.

In [21]:
def all_shapes(d, p):
    """Mọi hình dạng cây (tập các nút (d, p)) có gốc tại (d, p), độ sâu tối đa 4."""
    if d == 4:
        return [[(d, p)]]
    left = [[]] + all_shapes(d + 1, 2 * p - 1)
    right = [[]] + all_shapes(d + 1, 2 * p)
    return [[(d, p)] + L + R for L in left for R in right]


shapes = all_shapes(1, 1)
print("số hình dạng:", len(shapes))
assert len(shapes) == 676

rng = random.Random(666)
total = 0
for shape in shapes:
    for _ in range(3):
        nums = sorted(d * 100 + p * 10 + rng.randint(0, 9) for d, p in shape)
        expected = oracle_path_sum(nums)
        for name, fn in ALL.items():
            assert fn(nums) == expected, (name, nums)
        total += 1
print("đã kiểm", total, "cây (676 hình dạng × 3 bộ giá trị) với", len(ALL), "cách")

số hình dạng: 676


đã kiểm 2028 cây (676 hình dạng × 3 bộ giá trị) với 6 cách


### 12.4. Ngẫu nhiên có seed (xác suất có con khác nhau)

Sinh cây ngẫu nhiên: mỗi nút ở độ sâu `< 4` có mỗi con với xác suất `p_child`. Thử `p_child` từ thưa tới dày.

In [22]:
def random_tree(rng, p_child):
    nodes = {(1, 1): rng.randint(0, 9)}
    frontier = [(1, 1)]
    while frontier:
        d, p = frontier.pop()
        if d == 4:
            continue
        for q in (2 * p - 1, 2 * p):
            if rng.random() < p_child:
                nodes[(d + 1, q)] = rng.randint(0, 9)
                frontier.append((d + 1, q))
    return sorted(d * 100 + p * 10 + v for (d, p), v in nodes.items())


rng = random.Random(2024)
sizes = []
for _ in range(2000):
    nums = random_tree(rng, rng.choice([0.2, 0.4, 0.6, 0.8, 1.0]))
    assert 1 <= len(nums) <= 15 and all(110 <= x <= 489 for x in nums)
    assert nums == sorted(nums)
    expected = oracle_path_sum(nums)
    for name, fn in ALL.items():
        assert fn(nums) == expected, (name, nums)
    sizes.append(len(nums))
print("2000 cây ngẫu nhiên khớp | kích thước nhỏ nhất/lớn nhất:", min(sizes), max(sizes))

2000 cây ngẫu nhiên khớp | kích thước nhỏ nhất/lớn nhất:

 1 15


### 12.5. Cây đầy đủ với mọi giá trị `v` và cây toàn `9`

Cây đầy đủ 15 nút, mọi giá trị bằng `v`: có `8` lá, mỗi đường `4` nút → đáp án `32 * v`. Với `v = 9` đáp án lớn nhất có thể là `288`.

In [23]:
for v in range(10):
    nums = [d * 100 + p * 10 + v for d in range(1, 5) for p in range(1, 2 ** (d - 1) + 1)]
    for name, fn in ALL.items():
        assert fn(nums) == 32 * v, (name, v)
print("cây đầy đủ: đáp án 32 * v đúng với v = 0..9; lớn nhất =", 32 * 9)

# cây có chỉ một nhánh sâu và nhiều chỗ vắng
sparse = [119, 229, 349, 489]
print("nhánh phải dài:", [fn(sparse) for fn in ALL.values()])

cây đầy đủ: đáp án 32 * v đúng với v = 0..9; lớn nhất = 288
nhánh phải dài: [36, 36, 36, 36, 36, 36]


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi nút có **một** con là lá (dùng `or` thay vì `and`)

In [24]:
def wrong_leaf_or(nums):
    mp = {num // 10: num % 10 for num in nums}
    ans = 0

    def dfs(node, t):
        nonlocal ans
        if node not in mp:
            return
        t += mp[node]
        l, r = child_ids(node)
        if l not in mp or r not in mp:       # SAI: chỉ cần thiếu một con là coi như lá
            ans += t
            return
        dfs(l, t)
        dfs(r, t)

    dfs(11, 0)
    return ans


print("sai :", wrong_leaf_or([113, 221]), "| đúng:", path_sum_dfs([113, 221]))
assert wrong_leaf_or([113, 221]) == 3 and path_sum_dfs([113, 221]) == 4

sai :

 3 | đúng: 4


**Vì sao sai:** gốc `3` của `[113, 221]` thiếu con trái nên bị coi là lá: cộng `3` rồi dừng, bỏ sót con phải `1`. Lá phải thiếu **cả hai** con. **Cách đúng:** `if l not in mp and r not in mp`.

### Lỗi 2 — Sai công thức con (lệch một vị trí)

In [25]:
def wrong_child_formula(nums):
    mp = {num // 10: num % 10 for num in nums}
    ans = 0

    def dfs(node, t):
        nonlocal ans
        if node not in mp:
            return
        t += mp[node]
        d, p = divmod(node, 10)
        l = (d + 1) * 10 + 2 * p            # SAI: phải là 2p - 1
        r = l + 1
        if l not in mp and r not in mp:
            ans += t
            return
        dfs(l, t)
        dfs(r, t)

    dfs(11, 0)
    return ans


print("sai :", wrong_child_formula([113, 215, 221]), "| đúng:", path_sum_dfs([113, 215, 221]))
assert wrong_child_formula([113, 215, 221]) == 4 and path_sum_dfs([113, 215, 221]) == 12

sai : 4 | đúng: 12


**Vì sao sai:** con trái của `(d, p)` là vị trí `2p - 1`, không phải `2p`. Với `p = 1`: con là `1` và `2`, không phải `2` và `3`. Nhớ bằng ví dụ: tầng 2 có 2 vị trí (1 và 2) là con của vị trí 1 tầng 1. **Cách đúng:** `l = (d + 1) * 10 + 2 * p - 1; r = l + 1`.

### Lỗi 3 — Cộng tổng mọi giá trị hoặc chỉ giá trị ở lá (không phải tổng đường đi)

In [26]:
def wrong_sum_all(nums):
    return sum(num % 10 for num in nums)


def wrong_sum_leaf_values(nums):
    mp = {num // 10: num % 10 for num in nums}
    total = 0
    for key, v in mp.items():
        l, r = child_ids(key)
        if l not in mp and r not in mp:
            total += v
    return total


nums = [113, 215, 221]
print("tổng mọi giá trị :", wrong_sum_all(nums), "| tổng giá trị lá:", wrong_sum_leaf_values(nums), "| đúng:", path_sum_dfs(nums))
assert wrong_sum_all(nums) == 9 and wrong_sum_leaf_values(nums) == 6 and path_sum_dfs(nums) == 12

tổng mọi giá trị : 9 | tổng giá trị lá: 6 | đúng: 12


**Vì sao sai:** đáp án là **tổng của các tổng đường đi**; gốc `3` nằm trên **hai** đường nên được tính **hai lần** (`3 + 5` và `3 + 1`). Cộng mỗi nút một lần chỉ ra `9`. **Cách đúng:** mang theo `t` (tổng đường đi) khi xuống, hoặc dùng `giá trị × số lá trong cây con` (cách 5).

### Lỗi 4 — Dùng biến toàn cục cho tổng đường đi mà không quay lui

In [27]:
def wrong_no_backtrack(nums):
    mp = {num // 10: num % 10 for num in nums}
    ans = 0
    cur = 0                                   # SAI: biến dùng chung cho mọi nhánh

    def dfs(node):
        nonlocal ans, cur
        if node not in mp:
            return
        cur += mp[node]
        l, r = child_ids(node)
        if l not in mp and r not in mp:
            ans += cur
            return                            # SAI: thoát mà không trừ lại mp[node]
        dfs(l)
        dfs(r)
        cur -= mp[node]

    dfs(11)
    return ans


print("sai :", wrong_no_backtrack([113, 215, 221]), "| đúng:", path_sum_dfs([113, 215, 221]))
assert wrong_no_backtrack([113, 215, 221]) == 17 and path_sum_dfs([113, 215, 221]) == 12

sai : 17 | đúng: 12


**Vì sao sai:** sau khi xử lý lá `5`, biến `cur` vẫn còn `8`; sang nhánh phải `1` ta được `9` thay vì `4`. Khi dùng **biến chung** phải **trừ lại** (quay lui — backtracking) trước mọi lệnh `return`. **Cách đúng:** truyền `t` qua tham số (mỗi lời gọi có bản sao riêng) như `path_sum_dfs`, hoặc trừ lại đầy đủ.

### Lỗi 5 — Nhầm khóa: giữ cả chữ số giá trị trong khóa

In [28]:
def wrong_key_includes_value(nums):
    mp = {num: num % 10 for num in nums}      # SAI: khóa 113 thay vì 11
    ans = 0

    def dfs(node, t):
        nonlocal ans
        if node not in mp:
            return
        t += mp[node]
        l, r = child_ids(node)
        if l not in mp and r not in mp:
            ans += t
            return
        dfs(l, t)
        dfs(r, t)

    dfs(11, 0)
    return ans


print("sai :", wrong_key_includes_value([113, 215, 221]), "| đúng:", path_sum_dfs([113, 215, 221]))
assert wrong_key_includes_value([113, 215, 221]) == 0

sai : 0 | đúng: 12


**Vì sao sai:** ta tìm mã nút `11` (2 chữ số) nhưng khóa là `113` (3 chữ số), nên `11 not in mp` ngay từ đầu và trả `0`. **Cách đúng:** khóa là `num // 10` (bỏ chữ số giá trị).

### Lỗi 6 (bonus) — Quên rằng vị trí `p` tính theo cây ĐẦY ĐỦ

Khi một nút bị vắng, **vị trí vẫn được giữ chỗ**. Ví dụ `[113, 221]`: nút `22` nằm ở vị trí 2 dù vị trí 1 vắng. Nếu bạn "dồn" lại thành vị trí 1 thì sai. Công thức con/cha luôn dùng vị trí trong cây đầy đủ.

In [29]:
nums = [113, 221, 335]            # 22 -> con trái 33 (vị trí 3 tầng 3), 33 nằm dưới 22
mp = {num // 10: num % 10 for num in nums}
print("con của 22:", child_ids(22), "| có 33?", 33 in mp)
print("tổng:", path_sum_dfs(nums), "| oracle:", oracle_path_sum(nums))
assert path_sum_dfs(nums) == oracle_path_sum(nums) == 3 + 1 + 5

con của 22: (33, 34) | có 33? True
tổng: 9 | oracle: 9


## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra bài "cây mã hóa dưới dạng mảng":**

- Đề cho độ sâu nhỏ, nút ghi bằng số nhiều chữ số → **đừng dựng cây**, hãy dùng **dict mã → giá trị** và công thức con/cha.
- "Tổng mọi đường gốc → lá" → DFS truyền tổng xuống, chốt tại lá; hoặc đếm đóng góp `giá trị × số lá`.

**Mẹo nhớ:**

1. Con của `(d, p)`: `(d+1, 2p-1)` và `(d+1, 2p)`; cha: `(d-1, (p+1)//2)`.
2. Lá ⇔ **cả hai** con vắng → dùng `and`.
3. Truyền tổng bằng **tham số** để khỏi phải quay lui.
4. `nums` đã tăng dần → cha luôn đứng trước con: có thể quét xuôi một lượt.
5. Để tránh nhầm công thức: tự vẽ cây 3 tầng và kiểm tra vị trí 1 → con 1, 2; vị trí 2 → con 3, 4.
6. Muốn kiểm tra độc lập: đánh số kiểu heap `2^(d-1) + p - 1` (con `2h`, `2h+1`, cha `h//2`).
7. Họ hàng: LeetCode 112/113 (Path Sum I/II), 437 (Path Sum III), 129 (Sum Root to Leaf Numbers), 124 (Binary Tree Maximum Path Sum).

## 15. Biến thể

Dùng chung một hàm liệt kê đường đi từ oracle (đánh số heap) làm chuẩn để kiểm chứng các biến thể.

### 15.1. Có tồn tại đường gốc → lá có tổng bằng `target`? (kiểu LeetCode 112)

In [30]:
def has_path_sum(nums, target):
    mp = {num // 10: num % 10 for num in nums}

    def dfs(node, t):
        if node not in mp:
            return False
        t += mp[node]
        l, r = child_ids(node)
        if l not in mp and r not in mp:
            return t == target
        return dfs(l, t) or dfs(r, t)

    return dfs(11, 0)


tree5 = [113, 215, 221, 314, 326]
print([has_path_sum(tree5, t) for t in (12, 14, 4, 30, 0)])
assert [has_path_sum(tree5, t) for t in (12, 14, 4, 30, 0)] == [True, True, True, False, False]
rng = random.Random(151)
for _ in range(500):
    nums = random_tree(rng, 0.7)
    for target in range(0, 40, 3):
        assert has_path_sum(nums, target) == any(sum(p) == target for p in oracle_paths(nums))
print("15.1 OK")

[True, True, True, False, False]


15.1 OK


### 15.2. Đường gốc → lá có tổng **lớn nhất**

In [31]:
def max_root_leaf_sum(nums):
    mp = {num // 10: num % 10 for num in nums}

    def dfs(node):
        l, r = child_ids(node)
        kids = [c for c in (l, r) if c in mp]
        if not kids:
            return mp[node]
        return mp[node] + max(dfs(c) for c in kids)

    return dfs(11)


assert max_root_leaf_sum(tree5) == 14
rng = random.Random(152)
for _ in range(500):
    nums = random_tree(rng, 0.7)
    assert max_root_leaf_sum(nums) == max(sum(p) for p in oracle_paths(nums))
print("15.2 OK:", max_root_leaf_sum(tree5))

15.2 OK: 14


### 15.3. Ghép chữ số dọc đường (kiểu LeetCode 129): mỗi đường là một số

Mỗi đường gốc → lá đọc thành một số bằng cách ghép các chữ số: đường `3 → 5 → 4` thành `354`. Chỉ cần đổi `t += v` thành `t = t * 10 + v`.

In [32]:
def sum_root_to_leaf_numbers(nums):
    mp = {num // 10: num % 10 for num in nums}
    ans = 0

    def dfs(node, t):
        nonlocal ans
        if node not in mp:
            return
        t = t * 10 + mp[node]
        l, r = child_ids(node)
        if l not in mp and r not in mp:
            ans += t
            return
        dfs(l, t)
        dfs(r, t)

    dfs(11, 0)
    return ans


print(sum_root_to_leaf_numbers(tree5))     # 354 + 356 + 31 = 741
assert sum_root_to_leaf_numbers(tree5) == 354 + 356 + 31
rng = random.Random(153)
for _ in range(500):
    nums = random_tree(rng, 0.7)
    expected = sum(int("".join(map(str, p))) for p in oracle_paths(nums))
    assert sum_root_to_leaf_numbers(nums) == expected
print("15.3 OK")

741
15.3 OK


### 15.4. Phép ngược: mã hóa `TreeNode` thành mảng 3 chữ số

Ta có thể chuyển cây `TreeNode` (độ sâu `<= 4`, giá trị `0..9`) về dạng mã hóa: duyệt cây, mỗi nút `(d, p)` ghi `d*100 + p*10 + val`, con trái `(d+1, 2p-1)`, con phải `(d+1, 2p)`. Kiểm tra vòng khứ hồi `nums → cây → nums`.

In [33]:
def encode_tree(root):
    out = []

    def go(node, d, p):
        if node is None:
            return
        out.append(d * 100 + p * 10 + node.val)
        go(node.left, d + 1, 2 * p - 1)
        go(node.right, d + 1, 2 * p)

    go(root, 1, 1)
    return sorted(out)


assert encode_tree(build_tree([113, 215, 221])) == [113, 215, 221]
rng = random.Random(154)
for _ in range(500):
    nums = random_tree(rng, 0.7)
    assert encode_tree(build_tree(nums)) == nums
print("15.4 OK: nums -> cây -> nums giữ nguyên")

15.4 OK: nums -> cây -> nums giữ nguyên


### 15.5. Đếm số lá và độ sâu lớn nhất

Hai đại lượng thường gặp khi làm việc với cây mã hóa.

In [34]:
def count_leaves(nums):
    mp = {num // 10 for num in nums}
    return sum(1 for key in mp if all(c not in mp for c in child_ids(key)))


def max_depth(nums):
    return max(num // 100 for num in nums)


assert count_leaves([113, 215, 221]) == 2 and max_depth([113, 215, 221]) == 2
assert count_leaves([113, 221]) == 1
rng = random.Random(155)
for _ in range(300):
    nums = random_tree(rng, 0.7)
    assert count_leaves(nums) == len(oracle_paths(nums))
print("15.5 OK:", count_leaves(tree5), max_depth(tree5))

15.5 OK: 3 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng `TreeNode` + DFS | đổi mã hóa thành cây thật | `O(n)` | `O(n)` | khi cần dùng lại thư viện/hàm cây có sẵn |
| 2. DFS trên dict | dict mã → giá trị, công thức con | `O(n)` | `O(n)` | **cách nộp**, ngắn gọn |
| 3. DFS lặp bằng ngăn xếp | stack `(mã, tổng)` | `O(n)` | `O(n)` | tránh giới hạn đệ quy |
| 4. Quét xuôi theo cha | `acc[con] = acc[cha] + v`, lá = chưa là cha | `O(n)` | `O(n)` | khi `nums` đã sắp xếp, không cần đệ quy |
| 5. Quét ngược đếm lá | `Σ v × số lá cây con` | `O(n)` | `O(n)` | góc nhìn "đóng góp", mở rộng sang nhiều bài |

Khuyến nghị: nộp **cách 2**; dùng cách 4/5 để hiểu sâu; dùng `oracle_path_sum` (đánh số heap) khi tự viết biến thể.

In [35]:
rng = random.Random(16)
for _ in range(500):
    nums = random_tree(rng, rng.choice([0.3, 0.6, 0.9]))
    res = {name: fn(nums) for name, fn in ALL.items()}
    res["oracle"] = oracle_path_sum(nums)
    assert len(set(res.values())) == 1, (nums, res)
print("500 cây: 7 hàm cho cùng đáp án")

500 cây: 7 hàm cho cùng đáp án


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print((215 // 100, 215 // 10 % 10, 215 % 10))
print(child_ids(22))
print(path_sum_dfs([113, 221]))
```

In [36]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `nums = [113, 215, 221, 314, 326]`: (a) liệt kê **mã nút của các lá**; (b) liệt kê tổng của từng đường gốc → lá; (c) đáp án cuối cùng.

In [37]:
# Bài 2: điền kết quả tính tay
hand_leaf_ids = None        # danh sách mã nút các lá, ví dụ [22, 31]
hand_path_sums = None       # danh sách tổng từng đường
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành DFS trên dict.

In [38]:
def exercise_3_path_sum(nums: List[int]) -> int:
    mp = {num // 10: num % 10 for num in nums}
    ans = 0

    def dfs(node, t):
        nonlocal ans
        # TODO 1: nếu node không có trong mp thì return
        # TODO 2: cộng mp[node] vào t
        # TODO 3: tính mã hai con (con trái = (d + 1) * 10 + 2 * p - 1, con phải = trái + 1)
        # TODO 4: nếu CẢ HAI con đều vắng thì cộng t vào ans rồi return
        # TODO 5: gọi dfs cho hai con
        pass

    dfs(11, 0)
    return None


# assert exercise_3_path_sum([113, 215, 221]) == 12
# assert exercise_3_path_sum([113, 221]) == 4
# assert exercise_3_path_sum([113, 215, 221, 314, 326]) == 30

### Bài 4 — Tự viết lại (vừa-khó): quét xuôi theo cha

Không nhìn lại cách 4, hãy tự viết `exercise_4_parent(nums)`: duyệt `nums` theo thứ tự cho sẵn, tính `acc[key]` = tổng đường tới nút, đánh dấu cha, cuối cùng cộng `acc` của các nút chưa là cha.

```python
exercise_4_parent([113, 215, 221]) -> 12
exercise_4_parent([113, 221]) -> 4
exercise_4_parent([113]) -> 3
```

In [39]:
def exercise_4_parent(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_parent([113, 215, 221]) == 12
# assert exercise_4_parent([113, 221]) == 4
# assert exercise_4_parent([113]) == 3

### Bài 5 — Mini challenge: đếm số đường gốc → lá có tổng đúng bằng `target` (khó)

Viết `exercise_5_count_paths(nums, target)` trả về **số** đường gốc → lá có tổng đúng bằng `target`.

```python
exercise_5_count_paths([113, 215, 221], 8)  -> 1
exercise_5_count_paths([113, 215, 221], 12) -> 0
exercise_5_count_paths([113, 215, 221, 314, 326], 4) -> 1
exercise_5_count_paths([114, 214, 224, 314, 324, 334, 344], 12) -> 4   (cây 3 tầng đầy đủ toàn số 4)
```

Gợi ý hướng nghĩ: dùng DFS truyền tổng `t`; tại lá nếu `t == target` thì đếm.

In [40]:
def exercise_5_count_paths(nums: List[int], target: int) -> int:
    # TODO
    pass


# assert exercise_5_count_paths([113, 215, 221], 8) == 1
# assert exercise_5_count_paths([113, 215, 221], 12) == 0
# assert exercise_5_count_paths([113, 215, 221, 314, 326], 4) == 1

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `//` chia nguyên, `%` lấy dư; `child_ids(key)` dùng công thức `2p - 1`.
- Mức 2: `22` là tầng 2, vị trí 2 → con ở tầng 3, vị trí `2*2-1 = 3` và `4`. Ví dụ 2 là gốc chỉ có con phải: một đường.
- Mức 3: dòng 1 là `(2, 1, 5)`; dòng 2 là `(33, 34)`; dòng 3 là `4`.

**Bài 2**
- Mức 1: vẽ cây bằng `draw(nums)`: gốc `3` (mã 11); `5` (21), `1` (22); `4` (31), `6` (32).
- Mức 2: lá là nút không có con: `22`, `31`, `32`.
- Mức 3: (a) `[22, 31, 32]`; (b) `[4, 12, 14]` (hoặc thứ tự khác: `3+5+4 = 12`, `3+5+6 = 14`, `3+1 = 4`); (c) `30`.

**Bài 3**
- Mức 1: giống cách 2 ở mục 7.1.
- Mức 2: `if node not in mp: return`; `t += mp[node]`; `d, p = divmod(node, 10)`; `l = (d + 1) * 10 + 2 * p - 1`; `r = l + 1`.
- Mức 3: `if l not in mp and r not in mp: ans += t; return`; rồi `dfs(l, t)`, `dfs(r, t)`; cuối hàm `return ans` (không phải `None`).

**Bài 4**
- Mức 1: `acc = {}` và `has_child = set()`.
- Mức 2: với `d == 1` gán `acc[key] = v`; ngược lại `parent = (d - 1) * 10 + (p + 1) // 2`, `acc[key] = acc[parent] + v`, thêm `parent` vào `has_child`.
- Mức 3: đáp án `sum(acc[k] for k in acc if k not in has_child)`.

**Bài 5**
- Mức 1: sửa cách 2: thay `ans += t` thành "nếu `t == target` thì `ans += 1`".
- Mức 2: vẫn truyền `t` bằng tham số để không phải quay lui.
- Mức 3: kiểm tra bằng `sum(1 for p in oracle_paths(nums) if sum(p) == target)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [41]:
print((215 // 100, 215 // 10 % 10, 215 % 10))
print(child_ids(22))
print(path_sum_dfs([113, 221]))

assert (215 // 100, 215 // 10 % 10, 215 % 10) == (2, 1, 5)
assert child_ids(22) == (33, 34)
assert path_sum_dfs([113, 221]) == 4

(2, 1, 5)
(33, 34)
4


### Đáp án Bài 2

Cây: gốc `3` → con trái `5` (có hai con lá `4`, `6`), con phải `1` (lá).

| lá (mã) | đường | tổng |
|---|---|---|
| `31` | `3 → 5 → 4` | 12 |
| `32` | `3 → 5 → 6` | 14 |
| `22` | `3 → 1` | 4 |

Tổng `12 + 14 + 4 = 30`.

In [42]:
nums = [113, 215, 221, 314, 326]
draw(nums)
leaf_ids = sorted(k for k in {n // 10 for n in nums} if all(c not in {n // 10 for n in nums} for c in child_ids(k)))
sums = sorted(sum(p) for p in oracle_paths(nums))
print("lá:", leaf_ids, "| tổng đường:", sums, "| đáp án:", sum(sums))
assert leaf_ids == [22, 31, 32]
assert sums == [4, 12, 14]
assert sum(sums) == 30 == path_sum_dfs(nums)
print("Bài 2 OK")

tầng 1: 3
tầng 2: 5 1
tầng 3: 4 6 . .
lá: [22, 31, 32] | tổng đường: [4, 12, 14] | đáp án: 30
Bài 2 OK


### Đáp án Bài 3

In [43]:
def exercise_3_path_sum_solution(nums: List[int]) -> int:
    mp = {num // 10: num % 10 for num in nums}
    ans = 0

    def dfs(node, t):
        nonlocal ans
        if node not in mp:
            return
        t += mp[node]
        d, p = divmod(node, 10)
        l = (d + 1) * 10 + 2 * p - 1
        r = l + 1
        if l not in mp and r not in mp:
            ans += t
            return
        dfs(l, t)
        dfs(r, t)

    dfs(11, 0)
    return ans


assert exercise_3_path_sum_solution([113, 215, 221]) == 12
assert exercise_3_path_sum_solution([113, 221]) == 4
assert exercise_3_path_sum_solution([113, 215, 221, 314, 326]) == 30
rng = random.Random(3)
for _ in range(300):
    nums = random_tree(rng, 0.7)
    assert exercise_3_path_sum_solution(nums) == oracle_path_sum(nums)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [44]:
def exercise_4_parent_solution(nums: List[int]) -> int:
    acc = {}
    has_child = set()
    for num in nums:
        key, v = num // 10, num % 10
        d, p = divmod(key, 10)
        if d == 1:
            acc[key] = v
        else:
            parent = (d - 1) * 10 + (p + 1) // 2
            acc[key] = acc[parent] + v
            has_child.add(parent)
    return sum(a for k, a in acc.items() if k not in has_child)


assert exercise_4_parent_solution([113, 215, 221]) == 12
assert exercise_4_parent_solution([113, 221]) == 4
assert exercise_4_parent_solution([113]) == 3
rng = random.Random(4)
for _ in range(300):
    nums = random_tree(rng, 0.7)
    assert exercise_4_parent_solution(nums) == oracle_path_sum(nums)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [45]:
def exercise_5_count_paths_solution(nums: List[int], target: int) -> int:
    mp = {num // 10: num % 10 for num in nums}
    ans = 0

    def dfs(node, t):
        nonlocal ans
        if node not in mp:
            return
        t += mp[node]
        l, r = child_ids(node)
        if l not in mp and r not in mp:
            if t == target:
                ans += 1
            return
        dfs(l, t)
        dfs(r, t)

    dfs(11, 0)
    return ans


assert exercise_5_count_paths_solution([113, 215, 221], 8) == 1
assert exercise_5_count_paths_solution([113, 215, 221], 12) == 0
assert exercise_5_count_paths_solution([113, 215, 221, 314, 326], 4) == 1
full4 = [114, 214, 224, 314, 324, 334, 344]
print("ca tự kiểm tra:", exercise_5_count_paths_solution(full4, 12), "(3 tầng toàn 4 -> mọi đường 12, có 4 lá)")
assert exercise_5_count_paths_solution(full4, 12) == 4
rng = random.Random(5)
for _ in range(300):
    nums = random_tree(rng, 0.7)
    target = rng.randint(0, 30)
    assert exercise_5_count_paths_solution(nums, target) == sum(1 for p in oracle_paths(nums) if sum(p) == target)
print("Bài 5 OK")

ca tự kiểm tra:

 4 (3 tầng toàn 4 -> mọi đường 12, có 4 lá)
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
from typing import List

class Solution:
    def pathSum(self, nums: List[int]) -> int:
        def dfs(node, t):
            nonlocal ans
            if node not in mp:
                return
            t += mp[node]
            d, p = divmod(node, 10)
            l = (d + 1) * 10 + 2 * p - 1       # con trái: vị trí 2p - 1
            r = l + 1                          # con phải: vị trí 2p
            if l not in mp and r not in mp:    # lá: thiếu CẢ HAI con
                ans += t
                return
            dfs(l, t)
            dfs(r, t)

        ans = 0
        mp = {num // 10: num % 10 for num in nums}   # khóa = (độ sâu, vị trí)
        dfs(11, 0)
        return ans
```

Câu thần chú: **"khóa là `num // 10`, giá trị là `num % 10`; con là `2p - 1` và `2p`; lá phải thiếu cả hai con; truyền tổng xuống bằng tham số."**

### Checklist trước khi submit

- [ ] Khóa dict là `num // 10` (2 chữ số), không phải cả `num`.
- [ ] Công thức con: `(d + 1) * 10 + 2 * p - 1` và `+ 1`.
- [ ] Điều kiện lá dùng `and` (thiếu **cả hai** con).
- [ ] Tổng đường đi truyền qua tham số `t` (hoặc quay lui đầy đủ).
- [ ] Bắt đầu DFS từ mã gốc `11` với `t = 0`.
- [ ] Test: chỉ có gốc, gốc chỉ có một con, cây đầy đủ 15 nút (đáp án `32 * v` khi mọi giá trị bằng `v`).
- [ ] Bài Premium: nếu có tài khoản, đối chiếu lại đề gốc và hai hình ví dụ.
- [ ] Skeleton đúng: `class Solution`, method `pathSum(self, nums: List[int]) -> int`; độ phức tạp `O(n)`.